#  Big Data con PySpark — Notebook 2
## Selección, Filtros y Limpieza de Datos

---

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
# pyspark.sql.types → módulo con todos los tipos de datos de Spark
# Se usa para casteos explícitos y para definir esquemas manuales

spark = (
    SparkSession.builder
    .appName("Vuelos_Filtros")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("vuelos_colombia.csv")
)
df.cache()
df.count()  # materializa la caché
print("Datos listos")

Datos listos


---
## 1. Selección de columnas — `select()`

In [3]:
# ── Forma 1: por nombre (string) ──
df.select("aerolinea", "origen", "destino", "tarifa_usd").show(5)
# select(cols...) → devuelve un NUEVO DataFrame con solo esas columnas
# El original df no se modifica (DataFrames son inmutables en Spark)

+---------+------+-------+----------+
|aerolinea|origen|destino|tarifa_usd|
+---------+------+-------+----------+
|    LATAM|   LET|    PEI|     72.09|
|    LATAM|   CTG|    PEI|    478.33|
|    Wingo|   MDE|    VVC|    498.33|
|  EasyFly|   BAQ|    VVC|    247.92|
|  Avianca|   MTR|    BOG|    200.81|
+---------+------+-------+----------+
only showing top 5 rows


In [4]:
# ── Forma 2: con F.col() — más flexible ──
df.select(
    F.col("aerolinea"),                        # referencia directa a columna
    F.col("tarifa_usd").alias("precio"),        # rename con alias
    (F.col("distancia_km") / 1.60934).alias("distancia_millas"),  # operación matemática
).show(5)

+---------+------+------------------+
|aerolinea|precio|  distancia_millas|
+---------+------+------------------+
|    LATAM| 72.09|1228.4539003566679|
|    LATAM|478.33|379.65874209303195|
|    Wingo|498.33| 672.3253010550909|
|  EasyFly|247.92|1212.9195819404229|
|  Avianca|200.81|1091.1305255570608|
+---------+------+------------------+
only showing top 5 rows


In [5]:
# ── Excluir columnas con drop() ──
df_sin_id = df.drop("vuelo_id")
# drop(col) → devuelve el DF sin esa columna
# Útil para eliminar columnas de ID o redundantes
print("Columnas restantes:", df_sin_id.columns)

Columnas restantes: ['fecha', 'aerolinea', 'origen', 'destino', 'pasajeros', 'distancia_km', 'retraso_min', 'estado', 'tarifa_usd', 'clase']


---
## 2. Filtros — `filter()` / `where()`

In [6]:
# ── Filtro simple ──
vuelos_demorados = df.filter(F.col("estado") == "DEMORADO")
# filter(condicion) → mantiene solo las filas donde la condición es True
# .where() es exactamente igual → son aliases
print(f"Vuelos demorados: {vuelos_demorados.count():,}")

Vuelos demorados: 110,091


In [7]:
# ── Filtros combinados con & (AND) y | (OR) ──
# IMPORTANTE: cada condición debe ir entre paréntesis

vuelos_avianca_bog = df.filter(
    (F.col("aerolinea") == "Avianca") &   # AND lógico
    (F.col("origen") == "BOG")             # segunda condición
)
print(f"Avianca desde BOG: {vuelos_avianca_bog.count():,}")

vuelos_premium = df.filter(
    (F.col("clase") == "Business") |   # OR lógico
    (F.col("clase") == "Primera")
)
print(f"Vuelos premium: {vuelos_premium.count():,}")

Avianca desde BOG: 17,535
Vuelos premium: 124,232


In [8]:
# ── Filtro con isin() — equivalente al IN de SQL ──
rutas_principales = df.filter(
    F.col("origen").isin(["BOG", "MDE", "CLO"])  # el origen está en esta lista
)
print(f"Vuelos desde las 3 ciudades principales: {rutas_principales.count():,}")

Vuelos desde las 3 ciudades principales: 150,130


In [9]:
# ── Filtro por rango numérico ──
vuelos_cortos_baratos = df.filter(
    (F.col("distancia_km") < 500) &
    (F.col("tarifa_usd").between(50, 200))  # between es inclusivo en ambos extremos
)
vuelos_cortos_baratos.select("origen","destino","distancia_km","tarifa_usd").show(5)

+------+-------+------------+----------+
|origen|destino|distancia_km|tarifa_usd|
+------+-------+------------+----------+
|   BAQ|    MDE|         291|    100.91|
|   BOG|    VVC|         483|     52.68|
|   CLO|    CLO|         241|    189.03|
|   SMR|    VVC|         420|    117.73|
|   VVC|    MTR|         345|    184.23|
+------+-------+------------+----------+
only showing top 5 rows


In [10]:
# ── Filtrar nulos ──

# Solo filas donde tarifa NO es nula
df_con_tarifa = df.filter(F.col("tarifa_usd").isNotNull())
# isNotNull() → True cuando hay un valor (no es null)
# isNull()    → True cuando es null

# Solo filas donde retraso ES nulo
df_sin_retraso_dato = df.filter(F.col("retraso_min").isNull())

print(f"Con tarifa registrada:    {df_con_tarifa.count():,}")
print(f"Sin dato de retraso:      {df_sin_retraso_dato.count():,}")

Con tarifa registrada:    497,000
Sin dato de retraso:      5,000


---
## 3. Limpieza de datos

In [11]:
# ── Eliminar filas con nulos ──

# Elimina filas donde CUALQUIER columna tiene nulo
df_sin_nulos_total = df.dropna()

# Elimina filas donde TODAS las columnas son nulas
df_sin_nulos_all = df.dropna(how="all")

# Elimina filas con nulos solo en columnas específicas
df_limpio = df.dropna(subset=["tarifa_usd", "retraso_min", "clase"])
# subset → lista de columnas a revisar; solo elimina si esas columnas tienen nulo

print(f"Original:               {df.count():,}")
print(f"Sin CUALQUIER nulo:     {df_sin_nulos_total.count():,}")
print(f"Sin nulos en 3 cols:    {df_limpio.count():,}")

Original:               500,000
Sin CUALQUIER nulo:     491,039
Sin nulos en 3 cols:    491,039


In [12]:
# ── Rellenar nulos con un valor fijo ──
df_filled = df.fillna({
    "retraso_min": 0,          # si no hay dato de retraso, asumimos 0 minutos
    "tarifa_usd":  df.agg(F.avg("tarifa_usd")).first()[0],  # rellenar con el promedio
    "clase":       "Economica" # clase por defecto
})
# fillna(dict) → toma un diccionario col → valor de reemplazo
# .agg(F.avg(...)).first()[0] → calcula el promedio y extrae el número

# Verificar que ya no hay nulos en esas columnas
df_filled.select(
    F.count(F.when(F.col("retraso_min").isNull(), 1)).alias("nulos_retraso"),
    F.count(F.when(F.col("tarifa_usd").isNull(),  1)).alias("nulos_tarifa"),
    F.count(F.when(F.col("clase").isNull(),        1)).alias("nulos_clase"),
).show()

+-------------+------------+-----------+
|nulos_retraso|nulos_tarifa|nulos_clase|
+-------------+------------+-----------+
|            0|           0|          0|
+-------------+------------+-----------+



In [13]:
# ── Eliminar duplicados ──
df_unico = df.dropDuplicates()
# dropDuplicates() → elimina filas 100% iguales en todas las columnas

df_unico_por_vuelo = df.dropDuplicates(["vuelo_id"])
# dropDuplicates([cols]) → considera duplicado si esas columnas son iguales
# Útil cuando un vuelo_id aparece más de una vez por errores de ingesta

print(f"Original:            {df.count():,}")
print(f"Sin duplicados:      {df_unico.count():,}")

Original:            500,000
Sin duplicados:      500,000


In [14]:
# ── Casteo de tipos ──
# inferSchema a veces lee mal los tipos — es mejor definirlos explícitamente

df_typed = df.withColumn(
    "retraso_min",
    F.col("retraso_min").cast(IntegerType())
)
# withColumn(nombre, expresion):
#   - Si 'nombre' ya existe → REEMPLAZA la columna
#   - Si 'nombre' es nuevo  → AGREGA la columna
# .cast(Tipo()) → convierte el tipo de dato
# IntegerType() → entero 32 bits  (alternativa: LongType() = 64 bits)
# DoubleType()  → decimal de punto flotante
# StringType()  → texto

df_typed.printSchema()

root
 |-- vuelo_id: integer (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- aerolinea: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- destino: string (nullable = true)
 |-- pasajeros: integer (nullable = true)
 |-- distancia_km: integer (nullable = true)
 |-- retraso_min: integer (nullable = true)
 |-- estado: string (nullable = true)
 |-- tarifa_usd: double (nullable = true)
 |-- clase: string (nullable = true)



In [15]:
# ── Parsear fechas ──
df_con_fecha = df.withColumn(
    "fecha_ts",
    F.to_timestamp(F.col("fecha"), "yyyy-MM-dd HH:mm:ss")
    # to_timestamp(col, formato) → convierte string a TimestampType
    # yyyy = año 4 dígitos, MM = mes, dd = día, HH = hora 24h, mm = minutos, ss = segundos
)

# Extraer componentes de la fecha
df_fechas = df_con_fecha.select(
    "fecha",
    F.year("fecha_ts").alias("anio"),         # extrae el año
    F.month("fecha_ts").alias("mes"),          # extrae el mes (1-12)
    F.dayofweek("fecha_ts").alias("dia_sem"), # 1=domingo ... 7=sábado
    F.hour("fecha_ts").alias("hora"),          # extrae la hora (0-23)
)
df_fechas.show(5)

+-------------------+----+---+-------+----+
|              fecha|anio|mes|dia_sem|hora|
+-------------------+----+---+-------+----+
|2023-10-21 03:00:00|2023| 10|      7|   3|
|2022-02-05 20:00:00|2022|  2|      7|  20|
|2022-08-13 14:00:00|2022|  8|      7|  14|
|2023-05-14 12:00:00|2023|  5|      1|  12|
|2023-04-16 04:00:00|2023|  4|      1|   4|
+-------------------+----+---+-------+----+
only showing top 5 rows


---
## 4. Dataset limpio final

In [16]:
# Construimos el DataFrame limpio que usaremos en los siguientes notebooks
df_limpio_final = (
    df
    # 1. Eliminar filas con nulos en columnas críticas
    .dropna(subset=["tarifa_usd", "clase"])

    # 2. Rellenar nulos de retraso con 0
    .fillna({"retraso_min": 0})

    # 3. Eliminar vuelos donde origen == destino (datos inválidos)
    .filter(F.col("origen") != F.col("destino"))

    # 4. Parsear fecha y extraer componentes
    .withColumn("fecha_ts", F.to_timestamp("fecha", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("anio", F.year("fecha_ts"))
    .withColumn("mes",  F.month("fecha_ts"))
    .withColumn("hora", F.hour("fecha_ts"))

    # 5. Castear tipos
    .withColumn("retraso_min", F.col("retraso_min").cast(IntegerType()))

    # 6. Quitar columnas que ya no necesitamos
    .drop("fecha", "fecha_ts")
)

df_limpio_final.cache()
n_final = df_limpio_final.count()
print(f"Dataset limpio: {n_final:,} filas")
print(f"Filas eliminadas: {df.count() - n_final:,}")
df_limpio_final.printSchema()

Dataset limpio: 446,399 filas
Filas eliminadas: 53,601
root
 |-- vuelo_id: integer (nullable = true)
 |-- aerolinea: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- destino: string (nullable = true)
 |-- pasajeros: integer (nullable = true)
 |-- distancia_km: integer (nullable = true)
 |-- retraso_min: integer (nullable = true)
 |-- estado: string (nullable = true)
 |-- tarifa_usd: double (nullable = true)
 |-- clase: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- hora: integer (nullable = true)



In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Una vez montado el Drive, puedes guardar el archivo Parquet directamente en una carpeta de tu Drive, por ejemplo, en `Mi Drive/Colab Notebooks/`:

In [19]:
# Define la ruta en tu Google Drive donde quieres guardar el archivo
drive_path = '/content/drive/MyDrive/Colab Notebooks/vuelos_limpio.parquet'

# Guarda el DataFrame limpio como Parquet en Google Drive
(df_limpio_final
 .write
 .mode("overwrite")
 .parquet(drive_path))

print(f"Guardado como Parquet en Google Drive: {drive_path}")

Guardado como Parquet en Google Drive: /content/drive/MyDrive/Colab Notebooks/vuelos_limpio.parquet


---
## Resumen del notebook

```
select(cols)                     →  elegir columnas
drop(col)                        →  eliminar columna
filter(condicion)                →  filtrar filas
col.isin([lista])                →  equivalente a SQL IN
col.between(a, b)                →  rango inclusivo
col.isNull() / isNotNull()       →  detectar nulos
dropna(subset=[cols])            →  eliminar filas con nulos
fillna({col: valor})             →  rellenar nulos
dropDuplicates([cols])           →  eliminar duplicados
withColumn(nombre, expr)         →  crear/reemplazar columna
col.cast(Tipo())                 →  cambiar tipo de dato
to_timestamp(col, fmt)           →  parsear fecha
```

 **Siguiente:** Transformaciones — withColumn, when/otherwise, UDFs